# CV Phase 10 — Baseline training and resolution ablation

This notebook runs one controlled transfer-learning baseline at **512x512** and one resolution ablation at **224x224**. Both use the immutable patient-level physical split, the existing `SMCLUDDataset`, letterbox preprocessing, ImageNet normalization, no augmentation, the same ResNet18/ImageNet initialization, and the same optimizer/training policy.

The Test split is intentionally not loaded or evaluated. All charts, metrics, and image inspection remain inline in this notebook. No reports, plots, CSV/JSON summaries, or files under `results/cv/` are created.

## Audit and reproducibility configuration

There was no existing CV model implementation or checkpoint convention in the repository. This notebook therefore uses a single pretrained `torchvision.models.resnet18` with its final layer replaced by two outputs. The backbone is frozen and only the classification head is trained; this keeps the first baseline reproducible and limits the experiment to preprocessing/resolution effects.

In [ ]:
from pathlib import Path
import hashlib
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from torchvision.models import ResNet18_Weights, resnet18

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (CURRENT_DIR, *CURRENT_DIR.parents) if (p / 'src' / 'liversira_ai').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Project root containing src/liversira_ai was not found.')
ROOT = PROJECT_ROOT
sys.path.insert(0, str(ROOT / 'src'))
from liversira_ai.common.paths import CV_MANIFEST_DIR, CV_PROCESSED_DIR, CV_RAW_DIR
from liversira_ai.cv.data.preprocessing import LABELS, SPLITS, PreprocessingConfig, SMCLUDDataset, create_dataloader, load_image, resize_image
DATASET_ROOT = CV_PROCESSED_DIR / 'patient_split_v1'
METADATA_PATH = DATASET_ROOT / 'patient_split_metadata.csv'
SEED = 42
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HCC_INDEX = 0
HEMANGIOMA_INDEX = 1
print({'device': str(DEVICE), 'cuda_available': torch.cuda.is_available(), 'batch_size': BATCH_SIZE, 'max_epochs': MAX_EPOCHS, 'seed': SEED})

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
metadata = pd.read_csv(METADATA_PATH)
metadata['path'] = metadata['destination_rel_path'].map(lambda value: DATASET_ROOT / value)
metadata['source_path'] = metadata['source_rel_path'].map(lambda value: CV_RAW_DIR / value)
expected_counts = pd.DataFrame([
    {'split': 'train', 'HCC': 930, 'Hemangioma': 840, 'total': 1770},
    {'split': 'val', 'HCC': 332, 'Hemangioma': 297, 'total': 629},
    {'split': 'test', 'HCC': 127, 'Hemangioma': 130, 'total': 257},
])
actual_counts = metadata.pivot_table(index='split', columns='label', values='destination_rel_path', aggfunc='count', fill_value=0).reset_index()
actual_counts['total'] = actual_counts[['HCC', 'Hemangioma']].sum(axis=1)
display(actual_counts)
assert len(metadata) == 2656
assert actual_counts.set_index('split').loc[['train', 'val', 'test'], ['HCC', 'Hemangioma', 'total']].equals(expected_counts.set_index('split')[['HCC', 'Hemangioma', 'total']])
assert metadata['path'].map(Path.is_file).all()
assert metadata['source_path'].map(Path.is_file).all()
print('Physical split counts and filesystem paths verified; Test is not loaded by any training dataset.')

## Original dimensions, aspect ratios, and letterbox padding

This diagnostic uses Train and Validation only. Padding ratio is the fraction of target pixels that are padding after aspect-ratio-preserving resize. It is not used as a model feature; it is inspected for possible acquisition or class shortcuts.

In [ ]:
diagnostic = []
for row in metadata[metadata['split'].isin(['train', 'val'])].itertuples():
    image = load_image(row.path)
    scale = min(512 / image.width, 512 / image.height)
    resized_width = max(1, round(image.width * scale))
    resized_height = max(1, round(image.height * scale))
    content_fraction = (resized_width * resized_height) / (512 * 512)
    diagnostic.append({
        'split': row.split, 'label': row.label, 'path': row.path,
        'width': image.width, 'height': image.height,
        'aspect_ratio': image.width / image.height,
        'resized_width_512': resized_width, 'resized_height_512': resized_height,
        'padding_ratio_512': 1 - content_fraction,
    })
diagnostic = pd.DataFrame(diagnostic)
display(diagnostic.groupby(['split', 'label'])[['width', 'height', 'aspect_ratio', 'padding_ratio_512']].agg(['count', 'mean', 'median', 'min', 'max']).round(4))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for label in LABELS:
    subset = diagnostic[diagnostic['label'] == label]
    axes[0].hist(subset['width'], bins=20, alpha=0.55, label=label)
    axes[1].hist(subset['aspect_ratio'], bins=25, alpha=0.55, label=label)
    axes[2].hist(subset['padding_ratio_512'], bins=25, alpha=0.55, label=label)
axes[0].set_title('Original width'); axes[1].set_title('Aspect ratio'); axes[2].set_title('512 padding ratio')
for axis in axes: axis.legend(); axis.grid(alpha=0.2)
plt.tight_layout(); plt.show()
display(diagnostic.groupby(['split', 'label'])['padding_ratio_512'].mean().rename('mean_padding_ratio').to_frame().round(4))

In [ ]:
high_padding = diagnostic.sort_values('padding_ratio_512', ascending=False).head(4)
low_padding = diagnostic.sort_values('padding_ratio_512', ascending=True).head(4)
display(pd.concat([high_padding.assign(group='highest_padding'), low_padding.assign(group='lowest_padding')])[['group', 'split', 'label', 'width', 'height', 'aspect_ratio', 'padding_ratio_512', 'path']])
comparison_rows = pd.concat([high_padding.head(2), low_padding.head(2)])
fig, axes = plt.subplots(len(comparison_rows), 3, figsize=(10, 4 * len(comparison_rows)))
if len(comparison_rows) == 1: axes = np.array([axes])
for row_index, record in enumerate(comparison_rows.itertuples()):
    original = load_image(record.path)
    for axis, (title, image) in zip(axes[row_index], [('Original', original), ('Letterbox 512', resize_image(original, PreprocessingConfig((512, 512), 'letterbox'))), ('Letterbox 224', resize_image(original, PreprocessingConfig((224, 224), 'letterbox')))]):
        axis.imshow(image); axis.set_title(f'{title}\n{record.label}, padding={record.padding_ratio_512:.1%}'); axis.axis('off')
plt.tight_layout(); plt.show()
print('Interpretation: inspect whether padding differs systematically by class before treating it as a potential shortcut.')

## Model and metrics

Class mapping is explicit: `0 = HCC` (positive class), `1 = Hemangioma`. The same pretrained ResNet18 weights are loaded independently for both experiments. The backbone is frozen; only `fc` is optimized. No augmentation, class weighting, oversampling, or weighted sampler is used.

In [ ]:
def build_model():
    weights = ResNet18_Weights.DEFAULT
    model = resnet18(weights=weights)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.fc = torch.nn.Linear(model.fc.in_features, 2)
    return model.to(DEVICE)

def metric_row(y_true, probabilities, loss, train_loss=None):
    predictions = np.where(probabilities[:, HCC_INDEX] >= 0.5, HCC_INDEX, HEMANGIOMA_INDEX)
    truth = np.asarray(y_true, dtype=int)
    tn, fp, fn, tp = confusion_matrix(truth, predictions, labels=[0, 1]).ravel()
    return {
        'loss': float(loss), 'train_loss': None if train_loss is None else float(train_loss),
        'accuracy': accuracy_score(truth, predictions),
        'balanced_accuracy': balanced_accuracy_score(truth, predictions),
        'hcc_sensitivity': recall_score(truth, predictions, pos_label=HCC_INDEX, zero_division=0),
        'specificity': recall_score(truth, predictions, pos_label=HEMANGIOMA_INDEX, zero_division=0),
        'precision': precision_score(truth, predictions, pos_label=HCC_INDEX, zero_division=0),
        'f1': f1_score(truth, predictions, pos_label=HCC_INDEX, zero_division=0),
        'roc_auc': roc_auc_score(truth == HCC_INDEX, probabilities[:, HCC_INDEX]) if len(np.unique(truth)) == 2 else np.nan,
        'pr_auc': average_precision_score(truth == HCC_INDEX, probabilities[:, HCC_INDEX]) if len(np.unique(truth)) == 2 else np.nan,
        'false_negatives': int(fn), 'true_negatives': int(tn), 'false_positives': int(fp), 'true_positives': int(tp),
    }

def evaluate(model, loader, criterion):
    model.eval(); total_loss = 0.0; labels = []; probabilities = []
    with torch.no_grad():
        for images, batch_labels, _patient_ids in loader:
            images, batch_labels = images.to(DEVICE), batch_labels.to(DEVICE)
            logits = model(images); total_loss += criterion(logits, batch_labels).item() * len(images)
            labels.extend(batch_labels.cpu().numpy().tolist()); probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
    probabilities = np.concatenate(probabilities)
    return metric_row(labels, probabilities, total_loss / len(loader.dataset)), labels, probabilities

def train_experiment(size):
    set_seed()
    cfg = PreprocessingConfig((size, size), 'letterbox', normalization='imagenet')
    train_dataset = SMCLUDDataset(DATASET_ROOT, 'train', cfg)
    val_dataset = SMCLUDDataset(DATASET_ROOT, 'val', cfg)
    train_loader = create_dataloader(DATASET_ROOT, 'train', cfg, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = create_dataloader(DATASET_ROOT, 'val', cfg, batch_size=BATCH_SIZE, shuffle=False)
    model = build_model()
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW((parameter for parameter in model.parameters() if parameter.requires_grad), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = []; best = None; stale = 0; started = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train(); train_loss_sum = 0.0
        for images, labels, _patient_ids in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True); loss = criterion(model(images), labels); loss.backward(); optimizer.step()
            train_loss_sum += loss.item() * len(images)
        val_metrics, val_labels, val_probabilities = evaluate(model, val_loader, criterion)
        val_metrics['epoch'] = epoch; val_metrics['train_loss'] = train_loss_sum / len(train_loader.dataset); history.append(val_metrics)
        print({key: round(value, 4) if isinstance(value, float) else value for key, value in val_metrics.items() if key in ['epoch', 'train_loss', 'loss', 'accuracy', 'hcc_sensitivity', 'specificity', 'f1', 'roc_auc']})
        if best is None or val_metrics['loss'] < best['metrics']['loss']:
            best = {'metrics': dict(val_metrics), 'model_state': {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}, 'labels': val_labels, 'probabilities': val_probabilities}
            stale = 0
        else:
            stale += 1
            if stale >= EARLY_STOPPING_PATIENCE: break
    elapsed = time.perf_counter() - started
    return {'size': size, 'model': model, 'history': pd.DataFrame(history), 'best': best, 'seconds': elapsed, 'train_count': len(train_dataset), 'val_count': len(val_dataset), 'config': cfg}


## Experiment A and B

The two runs differ only in input resolution. Test is not instantiated, evaluated, or used for decisions. Best validation model states are retained in memory only.

In [ ]:
experiments = {}
for size in (512, 224):
    print(f'\nStarting {size}x{size} Letterbox experiment')
    experiments[size] = train_experiment(size)
print({size: {'epochs': len(result['history']), 'seconds': round(result['seconds'], 1), 'best_epoch': int(result['best']['metrics']['epoch'])} for size, result in experiments.items()})

## Training curves and validation metrics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for size, result in experiments.items():
    history = result['history']
    axes[0, 0].plot(history['epoch'], history['train_loss'], label=f'{size} train')
    axes[0, 0].plot(history['epoch'], history['loss'], '--', label=f'{size} val')
    axes[0, 1].plot(history['epoch'], history['accuracy'], label=str(size))
    axes[1, 0].plot(history['epoch'], history['hcc_sensitivity'], label=str(size))
    axes[1, 1].plot(history['epoch'], history['f1'], label=str(size))
for axis, title in zip(axes.flat, ['Loss', 'Accuracy', 'HCC sensitivity', 'HCC F1']):
    axis.set_title(title); axis.set_xlabel('Epoch'); axis.grid(alpha=0.2); axis.legend()
plt.tight_layout(); plt.show()

summary_rows = []
for size, result in experiments.items():
    metrics = result['best']['metrics']
    summary_rows.append({'resolution': f'{size}x{size}', 'resize': 'Letterbox', 'best_epoch': metrics['epoch'], 'best_val_loss': metrics['loss'], 'accuracy': metrics['accuracy'], 'balanced_accuracy': metrics['balanced_accuracy'], 'hcc_sensitivity': metrics['hcc_sensitivity'], 'specificity': metrics['specificity'], 'precision': metrics['precision'], 'f1': metrics['f1'], 'roc_auc': metrics['roc_auc'], 'pr_auc': metrics['pr_auc'], 'train_seconds': result['seconds'], 'train_count': result['train_count'], 'val_count': result['val_count'], 'generalization_gap': result['history'].iloc[-1]['train_loss'] - result['history'].iloc[-1]['loss']})
comparison = pd.DataFrame(summary_rows).set_index('resolution')
display(comparison.round(4))

In [ ]:
for size, result in experiments.items():
    metrics = result['best']['metrics']; labels = np.asarray(result['best']['labels']); probabilities = result['best']['probabilities']; predictions = np.where(probabilities[:, HCC_INDEX] >= 0.5, HCC_INDEX, HEMANGIOMA_INDEX)
    print(f'{size}x{size} classification report; HCC is positive class (index 0)')
    print(classification_report(labels, predictions, labels=[0, 1], target_names=['HCC', 'Hemangioma'], zero_division=0))
    print({'false_negatives': metrics['false_negatives'], 'false_positives': metrics['false_positives'], 'confusion_matrix': confusion_matrix(labels, predictions, labels=[0, 1]).tolist()})
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].imshow(confusion_matrix(labels, predictions, labels=[0, 1]), cmap='Blues'); axes[0].set_title(f'{size} confusion matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True'); axes[0].set_xticks([0, 1], ['HCC', 'Hemangioma']); axes[0].set_yticks([0, 1], ['HCC', 'Hemangioma'])
    fpr, tpr, _ = roc_curve(labels == HCC_INDEX, probabilities[:, HCC_INDEX]); axes[1].plot(fpr, tpr, label=f'AUC={metrics["roc_auc"]:.3f}'); axes[1].plot([0, 1], [0, 1], '--', color='gray'); axes[1].set_title(f'{size} ROC'); axes[1].legend()
    precision, recall, _ = precision_recall_curve(labels == HCC_INDEX, probabilities[:, HCC_INDEX]); axes[2].plot(recall, precision, label=f'AP={metrics["pr_auc"]:.3f}'); axes[2].set_title(f'{size} Precision-Recall'); axes[2].legend()
    for axis in axes: axis.grid(alpha=0.2)
    plt.tight_layout(); plt.show()

## Interpretation and final validation

Use the inline comparison above to decide whether 512 preserves useful information at an acceptable computational cost. Do not select a resolution from size alone: compare HCC sensitivity, specificity, PR-AUC, false negatives, validation loss, generalization gap, and training time together. The padding analysis should also be considered: a large class-dependent padding difference would be a warning that acquisition geometry could act as a shortcut.

This notebook intentionally does not evaluate Test, perform augmentation, compare architectures, tune hyperparameters, or write checkpoints/results. The physical split and original files remain the source of truth.

In [ ]:
assert set(experiments) == {512, 224}
assert all(result['train_count'] == 1770 and result['val_count'] == 629 for result in experiments.values())
assert all(torch.isfinite(torch.tensor(result['history'].select_dtypes(include=[np.number]).to_numpy())).all() for result in experiments.values())
assert all(result['config'].resize_mode == 'letterbox' and result['config'].normalization == 'imagenet' for result in experiments.values())
print({'completed_experiments': sorted(experiments), 'test_evaluated': False, 'augmentation': False, 'checkpoint_persisted': False, 'result_files_written': False})